In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np

def query_len(query):
    return len(query)

def count_sql_operations(operations, query):
   
    # Convert the query to uppercase to handle case insensitivity
    query = query.upper()

    # Initialize a counter to track operation counts
    operation_counts = Counter()

    # Loop through each operation type and count its occurrences
    for operation in operations:
        # Find all occurrences of each operation
        matches = re.findall(r'\b' + re.escape(operation) + r'\b', query)
        operation_counts[operation] = len(matches)

    return operation_counts
    
def check_clause_recurrence(target_operations_counts,predicted_operations_counts):
        
    # Matching counts
    matching_counts = sum(1 for key in predicted_operations_counts if predicted_operations_counts[key] == target_operations_counts[key])
    
    # Calculate the total number of clauses (operations)
    total_clauses = len(predicted_operations_counts)
    
    # Calculate the similarity score as the percentage of matching counts
    similarity_score = matching_counts / total_clauses
    
    # Print the results
    print(f"Matching Clause Counts: {matching_counts}")
    print(f"Total Clauses: {total_clauses}")
    print(f"Similarity Score: {similarity_score:.2f}")

    return matching_counts, total_clauses,similarity_score

In [ ]:
#############################################################################
######### CHANGE INPUT FOLDER HERE ##########################################
#############################################################################
directory_path = "C:/Research-Paper/PAPER-WORK-2024/SQL-Databse-Execution-Test-Dataset/"
###############################################################################
######### CHANGE INPUT FILENAME HERE ##########################################
###############################################################################
filename = 'Spider-Data-7'

file_path = filename+'.xlsx'
# Read the excel file
df = pd.read_excel(directory_path+file_path, engine='openpyxl')

In [ ]:
# Define a list of SQL operations we want to track
operations = ['SELECT', 'FROM', 'JOIN', 'WHERE', 'GROUP BY', 'ORDER BY', 'HAVING', 'LIMIT', 'INSERT', 'UPDATE', 'DELETE']
   
# get query length
df['spider_query_length'] = df['spider_query'].apply(query_len)
df['text2sql_query_length'] = df['text2sql_query'].apply(query_len)

df['query_len_match'] = df['spider_query_length'] == df['text2sql_query_length']

# Apply the count_sql_operations function to calculate 'target_operations_counts'
df['target_operations_counts'] = df['spider_query'].apply(lambda query: count_sql_operations(operations, query))
# Apply the count_sql_operations function to calculate 'predicted_operations_counts'
df['predicted_operations_counts'] = df['text2sql_query'].apply(lambda query: count_sql_operations(operations, query))

# Apply check_clause_recurrence to calculate matching counts, total clauses, and similarity score
df[['matching_counts', 'total_clauses', 'similarity_score']] = df.apply(
    lambda row: pd.Series(check_clause_recurrence(row['target_operations_counts'], row['predicted_operations_counts'])), axis=1
)

# 1. identify the differences between operations counts 
df['operations_counts_difference'] = df['predicted_operations_counts'] - df['target_operations_counts']

# 2. calculate number of operations differ 
df = df.assign(
    **{ops + '__difference': df.apply(lambda row: row['operations_counts_difference'].get(ops, 0) if isinstance(row['operations_counts_difference'], dict) else 0, axis=1) for ops in operations}
)

# 3. identify echoic_hallucination in query
df['echoic_hallucination'] = df['operations_counts_difference'].apply(
    lambda diff: True if isinstance(diff, dict) and max(diff.values(), default=0) > 5 else False
)

In [ ]:
from sql_metadata import Parser
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sortedcontainers import SortedDict

def query_parser(query):
    parse_dict = {}

    # lowercasing
    query = query.lower()

    try:
        # Extract tables
        tables = Parser(query).tables
        parse_dict['tables'] = sorted(tables)
    except KeyError as e:
        print(f"KeyError while extracting tables: {e}")
        parse_dict['tables'] = []  # Fallback to an empty list
    except Exception as e:
        print(f"Error while parsing tables: {e}")
        parse_dict['tables'] = []  # Fallback to an empty list
    
    try:
        # Extract columns using sql_metadata parser
        columns = Parser(query).columns
        parse_dict['columns'] = sorted(columns)
    except KeyError as e:
        print(f"KeyError while extracting columns: {e}")
        parse_dict['columns'] = []  # Fallback to an empty list if error occurs
    except Exception as e:
        print(f"Error while parsing columns: {e}")
        parse_dict['columns'] = []  # Fallback to an empty list
    
    try:
        # Extract operations
        operations = Parser(query).columns_dict
        parse_dict['operations'] = SortedDict(operations)
    except KeyError as e:
        print(f"KeyError while extracting operations: {e}")
        parse_dict['operations'] = {}  # Fallback to an empty dictionary
    except Exception as e:
        print(f"Error while parsing operations: {e}")
        parse_dict['operations'] = {}  # Fallback to an empty dictionary
    
    try:
        # Extract tables aliases
        tables_aliases = Parser(query).tables_aliases
        parse_dict['tables_aliases'] = SortedDict(tables_aliases)
    except KeyError as e:
        print(f"KeyError while extracting tables aliases: {e}")
        parse_dict['tables_aliases'] = {}  # Fallback to an empty dictionary
    except Exception as e:
        print(f"Error while parsing tables aliases: {e}")
        parse_dict['tables_aliases'] = {}  # Fallback to an empty dictionary

    return parse_dict

# Function to calculate Jaccard similarity for two sets
def jaccard_similarity(set1, set2):
    if len(set1) > 0 or len(set2) > 0:
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return intersection / union if union != 0 else 0
    else:
        return 1

# Function to compare r1 and r2
def compare_queries(r1, r2):
    similarity_score = {}

    # Columns comparison
    columns_r1 = set(r1['columns'])
    columns_r2 = set(r2['columns'])
    similarity_score['columns_similarity'] = jaccard_similarity(columns_r1, columns_r2)

    # Tables comparison
    tables_r1 = set(r1['tables'])
    tables_r2 = set(r2['tables'])
    similarity_score['tables_similarity'] = jaccard_similarity(tables_r1, tables_r2)

    # # Table Aliases comparison
    # aliases_r1 = set(r1['tables_aliases'].keys())
    # aliases_r2 = set(r2['tables_aliases'].keys())
    # similarity_score['aliases_similarity'] = jaccard_similarity(aliases_r1, aliases_r2)

    # Operations comparison (optional, for operation-based similarity)
    operations_r1 = set(r1['operations'].keys()) if r1.get('operations') else set()
    operations_r2 = set(r2['operations'].keys()) if r2.get('operations') else set()
    similarity_score['operations_similarity'] = jaccard_similarity(operations_r1, operations_r2)

    # Overall similarity (average of the individual similarities)
    overall_similarity = sum(similarity_score.values()) / len(similarity_score)
    similarity_score['overall_similarity'] = overall_similarity

    return similarity_score

def get_tables_aliases(parsed_query):
    return parsed_query['tables_aliases']

def normalised_query(query,alias_dict):
    #Lowercasing
    query = query.lower()

    #remove quotes
    query = query.replace('"', '').replace("'", '')
    
    # Iterate through the alias_dict and replace aliases in the query
    for alias, table in alias_dict.items():
        # Use regex to replace the alias with the table name in the query
        query = re.sub(rf'\b{alias}\b', table, query)
    return query    

def detect_hallucination(result1,result2):
    """
    hallucinations in selection
    """ 
    hallucination = False
    if (len(result1) > 0) and (len(result2) > 0):
        if len(set(result1) - set(result2)) > 0:
            hallucination = True
    return hallucination

In [ ]:
# Parse queries
df['parsed_spider_query'] = df['spider_query'].apply(query_parser)
df['parsed_text2sql_query'] = df['text2sql_query'].apply(query_parser)

# #################### extract tables_aliases ####################
# df['spider_query_tables_aliases'] = df['parsed_spider_query'].apply(get_tables_aliases)
# df['text2sql_query_tables_aliases'] = df['parsed_text2sql_query'].apply(get_tables_aliases)

#################### Normalise queries ##########################
df['normalised_spider_query'] = df.apply(lambda row: normalised_query(row['spider_query'], get_tables_aliases(row['parsed_spider_query'])), axis=1)
df['normalised_text2sql_query'] = df.apply(lambda row: normalised_query(row['text2sql_query'], get_tables_aliases(row['parsed_text2sql_query'])), axis=1)

#################### Compare the sql queries ####################
df['similarity_score'] = df.apply(lambda row: compare_queries(row['parsed_spider_query'], row['parsed_text2sql_query']), axis=1)



#################### extract table names ####################
df['spider_query_tables'] = df.apply(lambda row: row['parsed_spider_query']['tables'], axis=1)
df['text2sql_query_tables'] = df.apply(lambda row: row['parsed_text2sql_query']['tables'], axis=1)

# Store the tables_similarity score
df['tables_similarity'] = df.apply(lambda row: round(row['similarity_score']['tables_similarity'],2), axis=1)

# hallucinations in table selection
df['hallucinations_in_table_selection'] = df.apply(lambda row: detect_hallucination(row['spider_query_tables'], row['text2sql_query_tables']), axis=1)



#################### extract columns names ####################
df['spider_query_columns'] = df.apply(lambda row: row['parsed_spider_query']['columns'], axis=1)
df['text2sql_query_columns'] = df.apply(lambda row: row['parsed_text2sql_query']['columns'], axis=1)

# Store the columns_similarity score
df['columns_similarity'] = df.apply(lambda row: round(row['similarity_score']['columns_similarity'],2), axis=1)

# hallucinations in column selection
df['hallucinations_in_column_selection'] = df.apply(lambda row: detect_hallucination(row['spider_query_columns'], row['text2sql_query_columns']), axis=1)


    
#################### extract operations ####################
df['spider_query_operations'] = df.apply(lambda row: row['parsed_spider_query']['operations'], axis=1)
df['text2sql_query_operations'] = df.apply(lambda row: row['parsed_text2sql_query']['operations'], axis=1)

# Store the operations_similarity score
df['operations_similarity'] = df.apply(lambda row: round(row['similarity_score']['operations_similarity'],2), axis=1)

# hallucinations in operations selection
df['hallucinations_in_operations_selection'] = df.apply(lambda row: detect_hallucination(row['spider_query_operations'], row['text2sql_query_operations']), axis=1)


# hallucinations in query
df['query_similarity'] = df.apply(lambda row: round(row['similarity_score']['overall_similarity'],2), axis=1)
df['hallucinations_in_query'] = df[['hallucinations_in_table_selection','hallucinations_in_column_selection','hallucinations_in_operations_selection']].any(axis=1)

## Get SQL Query Execution Status

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine,text
import sqlparse
from typing import List

import pandas as pd

In [ ]:
def db_connection():
    # Replace with your PostgreSQL credentials
    username = "postgres"          # Default PostgreSQL username
    password = "123"     # Password for the 'postgres' user
    host = "localhost"             # Localhost if you're running PostgreSQL locally
    port = "5432"                  # Default PostgreSQL port
    database = "dev"     # The database name you want to connect to
    # Creating the database URL
    DATABASE_URL = f"postgresql://{username}:{password}@{host}:{port}/{database}"
    
    # Create an engine
    engine = create_engine(DATABASE_URL)
    print(f"Connected to PostgreSQL")
    return engine

def normalize_query(query: str) -> str:
    """
    Normalize the SQL query by formatting it consistently to enable comparison.
    """
    parsed = sqlparse.format(query, keyword_case="upper", reindent=True, strip_comments=True)
    return parsed
    
def execute_query(query: str,engine)  -> List[tuple]:
    """
    Execute the given SQL query on the database and fetch results.
    """
    status = False
    with engine.connect() as connection:
        try:
            result = connection.execute(text(query))
            
            # Fetch all rows from the result set
            rows = result.fetchall()
            status = True
            return rows ,status
        except:
            print(f"Error executing query")
            return [], status

def are_queries_equivalent(engine, target_query: str, predict_query: str) -> bool:
    """
    Check if two SQL queries produce the same results on the same database.
    """
    # Normalize the queries
    normalized_target_query = normalize_query(target_query)
    normalized_predict_query = normalize_query(predict_query)
    
    # Check if normalized queries are identical
    if normalized_target_query == normalized_predict_query:
        return True,0, 0,True,True
    else:
        # Otherwise, compare their outputs
        target_data,target_status = execute_query(engine,target_query)
        predicted_data,predicted_status = execute_query(engine,predict_query)
        return target_data == predicted_data,len(target_data), len(predicted_data),target_status,predicted_status

# Function to handle target and predicted queries based on target status
def process_queries(target_query, predicted_query, engine):
    # Step 1: identify result length and execution_status for target query
    target_data, target_execution_status = execute_query(target_query, engine)
    target_data_length = len(target_data)
    
    # Step 2: identify result length default 0 and execution_status with default False for predicted query if  target query execution_status is True 
    if target_execution_status:
        predicted_data,predicted_execution_status = execute_query(predicted_query, engine) 
        predicted_data_length = len(predicted_data)
    else:
        # If target query failed, set defaults for predicted query as well
        predicted_data = []
        predicted_data_length = 0 # Default value for predicted query if target status is True
        predicted_execution_status = False # Default status for predicted query if target status is True

    execution_result = target_data == predicted_data
    return target_data_length, target_execution_status, predicted_data_length, predicted_execution_status,execution_result

In [ ]:
engine = db_connection()

In [ ]:
# the normalization of target_query
df['normalized_target_query'] = df['spider_query'].apply(normalize_query)

df[['target_data_length', 'target_execution_status', 'predicted_data_length', 'predicted_execution_status','execution_result']] = df.apply(
    lambda row: pd.Series(process_queries(row['spider_query'], row['text2sql_query'], engine)), axis=1
)

In [ ]:
#############################################################################
######### CHANGE OUTPUT FOLDER FILE HERE ####################################
#############################################################################
output_file_path = filename+'__query_evaluation.xlsx'
df.to_excel(directory_path+ output_file_path, index=False)
print(f"DataFrame saved successfully to {directory_path+ output_file_path}")